# Method 1 — Part 2: pixel-wise linear heads (B0, B3, concatenation, noise control)

**Question:** with the simplest head (a logistic regression per pixel, no spatial context), does the concatenation TESSERA 2019 + AlphaEarth 2019 beat each embedding alone, and does it beat TESSERA + 64 channels of noise?

| Variant | In the plan |
|---|---|
| `t19` = TESSERA 2019 | baseline B0 |
| `ae19` = AlphaEarth 2019 | baseline B3 |
| `t19-ae19` = concatenation | M1-1 (concatenation), here with a linear head |
| `t19-noise` = TESSERA + 64 N(0, 1) channels | M1-2 (control for the extra dimensions) |

Same training pixels (300,000 valid pixels from the 1,455 training patches of fold I), same class weights (1/√frequency), per-channel standardisation, same evaluation on the 496 test patches (shared valid mask). An improvement is claimed only when the 95% interval of the paired bootstrap excludes 0. This is a first, linear look at H1a; the main Method 1 variants use the UNet head (after G1). For reference, *Better Together* found the concatenation worse than AlphaEarth alone on crops with a point-level linear head (57.5% vs 60.0%).

**Needs:** Parts 1–7 on the Drive. **Runtime:** CPU (High-RAM not needed). **Time:** about 15–30 min (copy of about 10.6 GB to the local disk, then 4 logistic regressions).

**Files needed in the repository** (new or changed in this part): `src/eval/metrics.py` (class names), `scripts/m1_linear_heads.py`, `notebooks/m1/02_linear_heads.ipynb`.

## 1. Runtime, Drive and configuration

In [ ]:
import sys, shutil, platform
from pathlib import Path
import pandas as pd
print("Python", platform.python_version())
print(f"free space in /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
RUNS = DRIVE / "runs"
RUNS_CSV = DRIVE / "results" / "runs.csv"
OUT = DRIVE / "results" / "m1"

## 2. Code and tests
**Done when:** all tests pass.

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests

## 3. Copy the data to the local disk
Reading the embeddings directly on the Drive is slow (plan, section 2): copy them once per session.

**Done when:** every line ends with `ok=True`.

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy", "ae_2019.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Train and evaluate the four linear heads
**Done when:** the summary table and the paired differences are printed, `results/runs.csv` holds four new runs (`lin_t19_100_foldI_s0`, `lin_ae19_...`, `lin_t19-ae19_...`, `lin_t19-noise_...`) and `results/m1/` holds `linear_heads_summary.csv`, `linear_heads_deltas.csv` and `linear_heads_per_class.png`.

**How to read it:**
- `t19-ae19` vs the better of `t19` and `ae19`: is the interval of the difference above 0?
- `t19-ae19` vs `t19-noise`: the concatenation must beat the same number of random channels, otherwise the gain only comes from the extra dimensions;
- `t19-noise` vs `t19` should be about 0 (or slightly negative: noise can only add overfitting);
- boundary vs core, and the IoU per class: where does AlphaEarth help or hurt?

In [ ]:
!python scripts/m1_linear_heads.py --processed "{PROC}" --runs-dir "{RUNS}" --runs-csv "{RUNS_CSV}" --out "{OUT}"

In [ ]:
from IPython.display import Image, display
display(Image(str(OUT / "linear_heads_per_class.png")))

## Results to report
1. The commit line and the test summary of step 2.
2. The training lines (features, iterations, seconds) and both tables of step 4.
3. The figure (paste it here).
4. Any error message, in full.